In [8]:
# # F1 Race Strategy Analysis — Undercut/Overcut Value by Constructor
#
# Question: How much does pit stop timing (early vs late relative to the field)
# change a driver's finishing position, and which constructors execute this best?
#
# Scope: 2011-2017 seasons — pit stop data is inconsistent/missing before this.

# ## 0. Find the correct dataset path
# Run this first — the folder name under /kaggle/input/ doesn't always match
# the dataset's URL slug exactly. Copy the printed path into DATA_DIR below.

# %%
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))

# ## 1. Imports

# %%
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go





/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/races.csv
/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/drivers.csv
/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/constructors.csv
/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/pitStops.csv
/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/status.csv
/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/seasons.csv
/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/constructorStandings.csv
/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/driverStandings.csv
/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/constructorResults.csv
/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/lapTimes.csv
/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/results.csv
/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/circuits.csv
/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/qualifying.csv


In [9]:
# ## 2. Set the data directory
# Paste the folder path (not a specific file) you saw printed in step 0,
# e.g. "/kaggle/input/formula-1-race-data-19502017/" — make sure it ends in a slash.
DATA_DIR = "/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/"

In [10]:
 ## 3. Load races.csv
 
# %%
races = pd.read_csv(DATA_DIR + "races.csv")
races.head()

,raceId,year,round,circuitId,name,date,time,url
0,1,2009,1,1,Australian Grand Prix,2009-03-29,06:00:00,http://en.wikipedia.org/wiki/2009_Australian_G...
1,2,2009,2,2,Malaysian Grand Prix,2009-04-05,09:00:00,http://en.wikipedia.org/wiki/2009_Malaysian_Gr...
2,3,2009,3,17,Chinese Grand Prix,2009-04-19,07:00:00,http://en.wikipedia.org/wiki/2009_Chinese_Gran...
3,4,2009,4,3,Bahrain Grand Prix,2009-04-26,12:00:00,http://en.wikipedia.org/wiki/2009_Bahrain_Gran...
4,5,2009,5,4,Spanish Grand Prix,2009-05-10,12:00:00,http://en.wikipedia.org/wiki/2009_Spanish_Gran...


In [11]:
# ## 4. Load results.csv
 
# %%
results = pd.read_csv(DATA_DIR + "results.csv")
results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22.0,1,1.0,1,1,10.0,58,34:50.6,5690616.0,39.0,2.0,01:27.5,218.3,1
1,2,18,2,2,3.0,5,2.0,2,2,8.0,58,5.478,5696094.0,41.0,3.0,01:27.7,217.586,1
2,3,18,3,3,7.0,7,3.0,3,3,6.0,58,8.163,5698779.0,41.0,5.0,01:28.1,216.719,1
3,4,18,4,4,5.0,11,4.0,4,4,5.0,58,17.181,5707797.0,58.0,7.0,01:28.6,215.464,1
4,5,18,5,1,23.0,3,5.0,5,5,4.0,58,18.014,5708630.0,43.0,1.0,01:27.4,218.385,1


In [12]:
# ## 5. Load pit_stops.csv
 
# %%
pit_stops = pd.read_csv(DATA_DIR + "pitStops.csv")
pit_stops.head()

,raceId,driverId,stop,lap,time,duration,milliseconds
0,841,153,1,1,17:05:23,26.898,26898
1,841,30,1,1,17:05:52,25.021,25021
2,841,17,1,11,17:20:48,23.426,23426
3,841,4,1,12,17:22:34,23.251,23251
4,841,13,1,13,17:24:10,23.842,23842


In [13]:
# ## 6. Load drivers.csv and constructors.csv
 
# %%
drivers = pd.read_csv(DATA_DIR + "drivers.csv", encoding="latin-1")
constructors = pd.read_csv(DATA_DIR + "constructors.csv", encoding="latin-1")

In [14]:
 ## 7. Load lap_times.csv
# Note: this file is large — this cell may take a moment.
 
# %%
lap_times = pd.read_csv(DATA_DIR + "lapTimes.csv")

In [15]:
# ## 8. Filter races to 2011-2017
 
# %%
races_scope = races[(races["year"] >= 2011) & (races["year"] <= 2017)][["raceId", "year", "round", "name", "circuitId"]]

In [16]:
 ## 9. Join results to race scope
 
# %%
results_clean = results.merge(races_scope, on="raceId", how="inner")
results_clean.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,year,round,name,circuitId
0,20779,841,20,9,1.0,1,1.0,1,1,25.0,...,5370259.0,44.0,4.0,01:29.8,212.488,1,2011,1,Australian Grand Prix,1
1,20780,841,1,1,3.0,2,2.0,2,2,18.0,...,5392556.0,41.0,8.0,01:30.3,211.382,1,2011,1,Australian Grand Prix,1
2,20781,841,808,4,10.0,6,3.0,3,3,15.0,...,5400819.0,55.0,7.0,01:30.1,211.969,1,2011,1,Australian Grand Prix,1
3,20782,841,4,6,5.0,5,4.0,4,4,12.0,...,5402031.0,49.0,2.0,01:29.5,213.336,1,2011,1,Australian Grand Prix,1
4,20783,841,17,9,2.0,3,5.0,5,5,10.0,...,5408430.0,50.0,3.0,01:29.6,213.066,1,2011,1,Australian Grand Prix,1


In [17]:
 ## 10. Add constructor and driver names to results
 
# %%
results_clean = results_clean.merge(
    constructors[["constructorId", "name"]].rename(columns={"name": "constructor"}),
    on="constructorId",
    how="left",
)
results_clean = results_clean.merge(
    drivers[["driverId", "surname"]], on="driverId", how="left"
)
results_clean.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,rank,fastestLapTime,fastestLapSpeed,statusId,year,round,name,circuitId,constructor,surname
0,20779,841,20,9,1.0,1,1.0,1,1,25.0,...,4.0,01:29.8,212.488,1,2011,1,Australian Grand Prix,1,Red Bull,Vettel
1,20780,841,1,1,3.0,2,2.0,2,2,18.0,...,8.0,01:30.3,211.382,1,2011,1,Australian Grand Prix,1,McLaren,Hamilton
2,20781,841,808,4,10.0,6,3.0,3,3,15.0,...,7.0,01:30.1,211.969,1,2011,1,Australian Grand Prix,1,Renault,Petrov
3,20782,841,4,6,5.0,5,4.0,4,4,12.0,...,2.0,01:29.5,213.336,1,2011,1,Australian Grand Prix,1,Ferrari,Alonso
4,20783,841,17,9,2.0,3,5.0,5,5,10.0,...,3.0,01:29.6,213.066,1,2011,1,Australian Grand Prix,1,Red Bull,Webber


In [18]:
# ## 11. Clean numeric columns
# positionOrder is a clean numeric finishing order (handles DNFs consistently)
 
# %%
results_clean["positionOrder"] = pd.to_numeric(
    results_clean["positionOrder"], errors="coerce"
)
results_clean["grid"] = pd.to_numeric(results_clean["grid"], errors="coerce")

In [19]:
 ## 12. Join pit stops to race scope
 
# %%
pit_stops_scope = pit_stops.merge(races_scope, on="raceId", how="inner")
pit_stops_scope["lap"] = pd.to_numeric(pit_stops_scope["lap"], errors="coerce")
pit_stops_scope.head()

,raceId,driverId,stop,lap,time,duration,milliseconds,year,round,name,circuitId
0,841,153,1,1,17:05:23,26.898,26898,2011,1,Australian Grand Prix,1
1,841,30,1,1,17:05:52,25.021,25021,2011,1,Australian Grand Prix,1
2,841,17,1,11,17:20:48,23.426,23426,2011,1,Australian Grand Prix,1
3,841,4,1,12,17:22:34,23.251,23251,2011,1,Australian Grand Prix,1
4,841,13,1,13,17:24:10,23.842,23842,2011,1,Australian Grand Prix,1


In [20]:
# ## 13. Get each driver's first pit stop per race
 
# %%
first_stops = (
    pit_stops_scope.sort_values(["raceId", "driverId", "lap"])
    .groupby(["raceId", "driverId"], as_index=False)
    .first()
)
first_stops.head()

,raceId,driverId,stop,lap,time,duration,milliseconds,year,round,name,circuitId
0,841,1,1,16,17:28:24,23.227,23227,2011,1,Australian Grand Prix,1
1,841,2,1,15,17:27:41,22.994,22994,2011,1,Australian Grand Prix,1
2,841,3,1,16,17:29:00,23.716,23716,2011,1,Australian Grand Prix,1
3,841,4,1,12,17:22:34,23.251,23251,2011,1,Australian Grand Prix,1
4,841,5,1,17,17:31:11,24.865,24865,2011,1,Australian Grand Prix,1


In [21]:
 ## 14. Calculate the field's median first-stop lap per race
# This is a rough proxy for "the field's" strategy window.
 
# %%
race_median_stop_lap = (
    first_stops.groupby("raceId")["lap"].median().rename("field_median_stop_lap")
)
first_stops = first_stops.merge(race_median_stop_lap, on="raceId", how="left")
first_stops.head()

,raceId,driverId,stop,lap,time,duration,milliseconds,year,round,name,circuitId,field_median_stop_lap
0,841,1,1,16,17:28:24,23.227,23227,2011,1,Australian Grand Prix,1,15.0
1,841,2,1,15,17:27:41,22.994,22994,2011,1,Australian Grand Prix,1,15.0
2,841,3,1,16,17:29:00,23.716,23716,2011,1,Australian Grand Prix,1,15.0
3,841,4,1,12,17:22:34,23.251,23251,2011,1,Australian Grand Prix,1,15.0
4,841,5,1,17,17:31:11,24.865,24865,2011,1,Australian Grand Prix,1,15.0


In [22]:
 ## 15. Classify each stop as undercut / overcut / inline
 
# %%
first_stops["stop_offset"] = first_stops["lap"] - first_stops["field_median_stop_lap"]
first_stops["strategy_type"] = np.select(
    [first_stops["stop_offset"] <= -2, first_stops["stop_offset"] >= 2],
    ["undercut", "overcut"],
    default="inline",
)
first_stops["strategy_type"].value_counts()

strategy_type
inline      1223
overcut      825
undercut     742
Name: count, dtype: int64

In [23]:
 ## 16. Join strategy calls to race outcomes
 
# %%
strategy_outcomes = first_stops.merge(
    results_clean[["raceId", "driverId", "constructor", "surname", "grid", "positionOrder"]],
    on=["raceId", "driverId"],
    how="inner",
)

strategy_outcomes.head()

,raceId,driverId,stop,lap,time,duration,milliseconds,year,round,name,circuitId,field_median_stop_lap,stop_offset,strategy_type,constructor,surname,grid,positionOrder
0,841,1,1,16,17:28:24,23.227,23227,2011,1,Australian Grand Prix,1,15.0,1.0,inline,McLaren,Hamilton,2,2
1,841,2,1,15,17:27:41,22.994,22994,2011,1,Australian Grand Prix,1,15.0,0.0,inline,Renault,Heidfeld,18,12
2,841,3,1,16,17:29:00,23.716,23716,2011,1,Australian Grand Prix,1,15.0,1.0,inline,Mercedes,Rosberg,7,17
3,841,4,1,12,17:22:34,23.251,23251,2011,1,Australian Grand Prix,1,15.0,-3.0,undercut,Ferrari,Alonso,5,4
4,841,5,1,17,17:31:11,24.865,24865,2011,1,Australian Grand Prix,1,15.0,2.0,overcut,Lotus,Kovalainen,19,18


In [24]:
# ## 17. Calculate positions gained/lost and drop incomplete rows
 
# %%
strategy_outcomes["positions_gained"] = (
    strategy_outcomes["grid"] - strategy_outcomes["positionOrder"]
)
strategy_outcomes = strategy_outcomes.dropna(subset=["grid", "positionOrder"])
strategy_outcomes.head()

,raceId,driverId,stop,lap,time,duration,milliseconds,year,round,name,circuitId,field_median_stop_lap,stop_offset,strategy_type,constructor,surname,grid,positionOrder,positions_gained
0,841,1,1,16,17:28:24,23.227,23227,2011,1,Australian Grand Prix,1,15.0,1.0,inline,McLaren,Hamilton,2,2,0
1,841,2,1,15,17:27:41,22.994,22994,2011,1,Australian Grand Prix,1,15.0,0.0,inline,Renault,Heidfeld,18,12,6
2,841,3,1,16,17:29:00,23.716,23716,2011,1,Australian Grand Prix,1,15.0,1.0,inline,Mercedes,Rosberg,7,17,-10
3,841,4,1,12,17:22:34,23.251,23251,2011,1,Australian Grand Prix,1,15.0,-3.0,undercut,Ferrari,Alonso,5,4,1
4,841,5,1,17,17:31:11,24.865,24865,2011,1,Australian Grand Prix,1,15.0,2.0,overcut,Lotus,Kovalainen,19,18,1


In [25]:
# ## 18. Aggregate strategy value by constructor
 
# %%
constructor_strategy = (
    strategy_outcomes.groupby(["constructor", "strategy_type"])["positions_gained"]
    .agg(avg_positions_gained="mean", stops="count")
    .reset_index()
)
constructor_strategy = constructor_strategy[constructor_strategy["stops"] >= 15]
constructor_strategy.head()

,constructor,strategy_type,avg_positions_gained,stops
0,Caterham,inline,2.634615,52
1,Caterham,overcut,2.043478,23
2,Caterham,undercut,1.541667,24
3,Ferrari,inline,0.482759,145
4,Ferrari,overcut,1.015873,63


In [26]:
# ## 19. Build the undercut leaderboard
 
# %%
undercut_leaderboard = constructor_strategy[
    constructor_strategy["strategy_type"] == "undercut"
].sort_values("avg_positions_gained", ascending=False)
 
undercut_leaderboard.head(10)

,constructor,strategy_type,avg_positions_gained,stops
26,Marussia,undercut,2.285714,21
2,Caterham,undercut,1.541667,24
41,Sauber,undercut,0.305263,95
29,McLaren,undercut,0.031250,64
8,Force India,undercut,-0.064103,78
14,Haas F1 Team,undercut,-0.272727,22
20,Lotus F1,undercut,-0.444444,45
50,Williams,undercut,-0.759494,79
35,Red Bull,undercut,-0.831169,77
38,Renault,undercut,-0.848485,33


In [35]:
# ## 20. Chart — undercut value by constructor
 
# %%
fig = px.bar(
    undercut_leaderboard.sort_values("avg_positions_gained"),
    x="avg_positions_gained",
    y="constructor",
    orientation="h",
    title="Average positions gained on undercut strategy by constructor (2011-2017)",
    labels={
        "avg_positions_gained": "Avg. positions gained",
        "constructor": "Constructor",
    },
)
fig.show()

In [36]:
 ## 21. Chart — undercut vs overcut spread
 
# %%
fig2 = px.box(
    strategy_outcomes[strategy_outcomes["strategy_type"] != "inline"],
    x="strategy_type",
    y="positions_gained",
    color="strategy_type",
    title="Positions gained: undercut vs overcut strategy calls (all constructors, 2011-2017)",
)
fig2.show()

In [37]:
# ## 22. Chart — season trend for selected constructors
 
# %%
season_trend = (
    strategy_outcomes[strategy_outcomes["strategy_type"] == "undercut"]
    .groupby(["year", "constructor"])["positions_gained"]
    .mean()
    .reset_index()
)
 
constructors_of_interest = ["Red Bull", "Mercedes", "Ferrari", "McLaren"]
fig3 = px.line(
    season_trend[season_trend["constructor"].isin(constructors_of_interest)],
    x="year",
    y="positions_gained",
    color="constructor",
    markers=True,
    title="Undercut strategy value by season, selected constructors",
)
fig3.show()

In [30]:
# ## 23. Export cleaned data for the dashboard
 
# %%
strategy_outcomes.to_csv("strategy_outcomes.csv", index=False)
constructor_strategy.to_csv("constructor_strategy_summary.csv", index=False)

In [31]:
# ## 24. Load circuits.csv for circuit names
 
# %%
circuits = pd.read_csv(DATA_DIR + "circuits.csv", encoding="cp1252")
circuits.head()

,circuitId,circuitRef,name,location,country,lat,lng,alt,url
0,1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.84970,144.96800,10.0,http://en.wikipedia.org/wiki/Melbourne_Grand_P...
1,2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.73800,NaN,http://en.wikipedia.org/wiki/Sepang_Internatio...
2,3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.03250,50.51060,NaN,http://en.wikipedia.org/wiki/Bahrain_Internati...
3,4,catalunya,Circuit de Barcelona-Catalunya,MontmelÌ_,Spain,41.57000,2.26111,NaN,http://en.wikipedia.org/wiki/Circuit_de_Barcel...
4,5,istanbul,Istanbul Park,Istanbul,Turkey,40.95170,29.40500,NaN,http://en.wikipedia.org/wiki/Istanbul_Park


In [32]:
 ## 25. Join circuit names onto strategy_outcomes
 
# %%
strategy_outcomes = strategy_outcomes.merge(
    circuits[["circuitId", "name"]].rename(columns={"name": "circuit"}),
    on="circuitId",
    how="left",
)
strategy_outcomes.head()

,raceId,driverId,stop,lap,time,duration,milliseconds,year,round,name,circuitId,field_median_stop_lap,stop_offset,strategy_type,constructor,surname,grid,positionOrder,positions_gained,circuit
0,841,1,1,16,17:28:24,23.227,23227,2011,1,Australian Grand Prix,1,15.0,1.0,inline,McLaren,Hamilton,2,2,0,Albert Park Grand Prix Circuit
1,841,2,1,15,17:27:41,22.994,22994,2011,1,Australian Grand Prix,1,15.0,0.0,inline,Renault,Heidfeld,18,12,6,Albert Park Grand Prix Circuit
2,841,3,1,16,17:29:00,23.716,23716,2011,1,Australian Grand Prix,1,15.0,1.0,inline,Mercedes,Rosberg,7,17,-10,Albert Park Grand Prix Circuit
3,841,4,1,12,17:22:34,23.251,23251,2011,1,Australian Grand Prix,1,15.0,-3.0,undercut,Ferrari,Alonso,5,4,1,Albert Park Grand Prix Circuit
4,841,5,1,17,17:31:11,24.865,24865,2011,1,Australian Grand Prix,1,15.0,2.0,overcut,Lotus,Kovalainen,19,18,1,Albert Park Grand Prix Circuit


In [33]:
# ## 26. Aggregate undercut value by circuit
# Some circuits (hard to overtake on track) reward undercuts far more than others.
 
# %%
circuit_strategy = (
    strategy_outcomes[strategy_outcomes["strategy_type"] == "undercut"]
    .groupby("circuit")["positions_gained"]
    .agg(avg_positions_gained="mean", stops="count")
    .reset_index()
)
 
# keep circuits with a reasonable sample size
circuit_strategy = circuit_strategy[circuit_strategy["stops"] >= 10]
circuit_strategy = circuit_strategy.sort_values("avg_positions_gained", ascending=False)
 
circuit_strategy.head(10)

,circuit,avg_positions_gained,stops
0,Albert Park Grand Prix Circuit,1.837838,37
1,Autodromo Nazionale di Monza,1.148936,47
8,Circuit de Barcelona-Catalunya,1.133333,30
24,Valencia Street Circuit,0.750000,12
17,NÌ_rburgring,0.647059,17
16,Marina Bay Street Circuit,0.444444,27
5,Baku City Circuit,0.250000,12
22,Sochi Autodrom,0.100000,30
7,Circuit Gilles Villeneuve,0.021277,47
3,AutÌ_dromo JosÌ© Carlos Pace,-0.083333,36


In [34]:
 ## 27. Chart — undercut value by circuit
 
# %%
fig4 = px.bar(
    circuit_strategy.sort_values("avg_positions_gained"),
    x="avg_positions_gained",
    y="circuit",
    orientation="h",
    title="Average positions gained on undercut strategy by circuit (2011-2017)",
    labels={
        "avg_positions_gained": "Avg. positions gained",
        "circuit": "Circuit",
    },
)
fig4.show()